In [16]:
import pandas as pd
import cv2 as cv
import os

~~Create Dataframe~~

Train, Evaluate, and Tune Model






In [ ]:
def load_data(split):
    df = pd.DataFrame()
    train_dir = '/Users/adharshv/Documents/Career/portfolio/Plant Disease/'+split.lower()+'/'
    dir_ls = os.listdir(train_dir)

    for dir in dir_ls:
        temp_df = pd.DataFrame()
        files = os.listdir(train_dir+dir)
        im_paths = [train_dir + dir+ '/' + i for i in files]
        temp_df['im_path'] = im_paths
        temp_df['plant_name'] = dir.split('_',1)[0]
        temp_df['disease_state'] = dir.split('_',1)[1].replace('_','')
        df = pd.concat([df,temp_df],ignore_index=True)
    return df

def plant_data(data,plant):
    return data[data['plant_name'] == plant]

raw_df = load_data('train')
raw_df
    

,im_path,plant_name,disease_state
0,/Users/adharshv/Documents/Career/portfolio/Pla...,Strawberry,healthy
1,/Users/adharshv/Documents/Career/portfolio/Pla...,Strawberry,healthy
2,/Users/adharshv/Documents/Career/portfolio/Pla...,Strawberry,healthy
3,/Users/adharshv/Documents/Career/portfolio/Pla...,Strawberry,healthy
4,/Users/adharshv/Documents/Career/portfolio/Pla...,Strawberry,healthy
...,...,...,...
70290,/Users/adharshv/Documents/Career/portfolio/Pla...,Soybean,healthy
70291,/Users/adharshv/Documents/Career/portfolio/Pla...,Soybean,healthy
70292,/Users/adharshv/Documents/Career/portfolio/Pla...,Soybean,healthy
70293,/Users/adharshv/Documents/Career/portfolio/Pla...,Soybean,healthy


In [1]:
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import os

In [2]:
class CustomImageDataset(Dataset):
    def __init__(self,img_dir,transform=None):
        self.img_dir = img_dir
        self.transform = transform
        self.img_labels = []
        
        for label, class_name in enumerate(os.listdir(img_dir)):
            class_dir = os.path.join(img_dir,class_name)
            if os.path.isdir(class_dir):
                for img_name in os.listdir(class_dir):
                    self.img_labels.append((os.path.join(class_dir,img_name),label))
                    
        self.classes = set([i[1] for i in self.img_labels])
                    
    def __len__(self):
        return len(self.img_labels)
    
    def __getitem__(self, index):
        img_path, label = self.img_labels[index]
        image = Image.open(img_path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        return image, label
    
    
    
class CNN(nn.Module):
    def __init__(self, num_classes):
        super(CNN,self).__init__()
        self.cnn_kernel = 3
        self.pool_kernel = 2
        self.padding = 1
        self.stride = 2
        self.conv1 = nn.Conv2d(3,16,kernel_size=self.cnn_kernel,padding=self.padding)
        self.pool = nn.MaxPool2d(kernel_size=self.pool_kernel, stride = self.stride)
        self.conv2 = nn.Conv2d(16,32,kernel_size=self.cnn_kernel,padding=self.padding)
        self.fc1 = nn.Linear(32*64*64,128)
        self.fc2 = nn.Linear(128,num_classes)
        
    def forward(self,x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = x.view(-1,32*64*64)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x
    



In [3]:
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))])

train_dataset = CustomImageDataset(img_dir = '/Users/adharshv/Documents/Career/portfolio/Plant Disease/train',transform=transform)
train_loader = DataLoader(train_dataset,batch_size=128,shuffle=True)

val_dataset = CustomImageDataset(img_dir = '/Users/adharshv/Documents/Career/portfolio/Plant Disease/valid',transform=transform)
val_loader = DataLoader(val_dataset,batch_size=64,shuffle=False)

model = CNN(len(train_dataset.classes))


criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(),lr = 0.001)

In [ ]:
#device = torch.device("cuda" if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')
device = torch.device('mps')
print(device)
model.to(device)


num_epochs = 5
loss_dct = {0:[]}

for epoch in range(num_epochs):
    model.train()
    for i, data in enumerate(train_loader,0):
        inputs, labels = data
        inputs, labels = inputs.to(device), labels.to(device)
        
        optimizer.zero_grad()
        
        outputs = model(inputs)
        loss = criterion(outputs.to(device), labels.to(device))
        loss.backward()
        optimizer.step()
        loss_dct[epoch].append(loss.item())
        
        if i % 10 == 0:
            print(f'Epoch {epoch + 1}, Loss: {loss.item()}')

mps
Epoch 1, Loss: 3.6366796493530273
Epoch 1, Loss: 3.441300868988037
Epoch 1, Loss: 2.9484775066375732
Epoch 1, Loss: 2.2234854698181152
Epoch 1, Loss: 2.0672640800476074
Epoch 1, Loss: 1.9325406551361084
Epoch 1, Loss: 1.559312343597412
Epoch 1, Loss: 1.3793996572494507
Epoch 1, Loss: 1.6869220733642578
Epoch 1, Loss: 1.350365400314331
Epoch 1, Loss: 1.2468816041946411
Epoch 1, Loss: 1.0706902742385864
Epoch 1, Loss: 0.8980765342712402
Epoch 1, Loss: 0.93287593126297
Epoch 1, Loss: 0.9264304637908936
Epoch 1, Loss: 1.1397621631622314
Epoch 1, Loss: 0.8195609450340271
Epoch 1, Loss: 0.8379242420196533
Epoch 1, Loss: 0.9664657711982727
Epoch 1, Loss: 0.8972042202949524
Epoch 1, Loss: 0.9088822603225708
Epoch 1, Loss: 0.8543568849563599
Epoch 1, Loss: 0.705345094203949
Epoch 1, Loss: 0.7328627109527588
Epoch 1, Loss: 0.6211804151535034
Epoch 1, Loss: 0.558212399482727
Epoch 1, Loss: 0.6281083822250366
Epoch 1, Loss: 0.6281739473342896
Epoch 1, Loss: 0.8609581589698792
Epoch 1, Loss: 0.

In [ ]:
torch.save(model.state_dict(), '/Users/adharshv/Documents/Career/portfolio/Plant Disease/models/modelv1.pt')